# BiDA Inference & Visualization
Run these cells to evaluate the trained model, generate the confusion matrix, and visualize the classification maps.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.io as io
from models.get_model import get_model
from utils.dataset import load_mat_hsi
from utils.utils_HSI import count_sliding_window, sliding_window, grouper, metrics
from tqdm.notebook import tqdm
import os


In [ ]:
# Define configurations
class Opts:
    pass

opts = Opts()
opts.model = 'BiDA'
opts.source_name = 'Houston13'
opts.target_name = 'Houston18'
opts.dataset_dir = './Houston/'
opts.patch_size = 13
opts.dim = 64
opts.depth = 3
opts.num_tokens = 4

# Check for MPS or CUDA
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda:0')
else:
    device = torch.device('cpu')

print(f'Using device: {device}')


In [ ]:
# Load target data
print('Loading dataset...')
img_tar, gt_tar, labels = load_mat_hsi(opts.target_name, opts.dataset_dir, norm='normband')
n_classes = len(labels)

print(f'Image shape: {img_tar.shape}')
print(f'Ground truth shape: {gt_tar.shape}')
print(f'Number of classes: {n_classes}')


In [ ]:
# Find the best checkpoint model
checkpoint_dir = f'./checkpoints/{opts.model}/{opts.source_name}to{opts.target_name}'
# Find the .pth file in the directory
model_files = [f for f in os.listdir(checkpoint_dir) if f.endswith('.pth')]
if not model_files:
    raise FileNotFoundError('No checkpoint found. Did training complete?')
best_model_file = sorted(model_files)[-1] # Gets the latest/best model
checkpoint_path = os.path.join(checkpoint_dir, best_model_file)
print(f'Loading model weights from: {checkpoint_path}')

# Initialize and load weights
network = get_model(opts.model, opts.source_name, opts.patch_size, opts)
network.load_state_dict(torch.load(checkpoint_path, map_location=device))
network.to(device)
network.eval()
print('Model loaded successfully.')


In [ ]:
# Sliding window inference
patch_size = opts.patch_size
batch_size = 128
window_size = (patch_size, patch_size)
image_w, image_h = img_tar.shape[:2]
pad_size = patch_size // 2

img_padded = np.pad(img_tar, ((pad_size, pad_size), (pad_size, pad_size), (0, 0)), mode='reflect')
probs = np.zeros(img_padded.shape[:2] + (n_classes, ))

iterations = count_sliding_window(img_padded, window_size=window_size) // batch_size

for batch in tqdm(grouper(batch_size, sliding_window(img_padded, window_size=window_size)), 
                  total=iterations, desc='Inference'):
    with torch.no_grad():
        data = [b[0] for b in batch]
        data = np.copy(data).transpose((0, 3, 1, 2))
        data = torch.from_numpy(data).float().unsqueeze(1).to(device)
        indices = [b[1:] for b in batch]
        
        # BiDA expects 2 inputs during training, but only target matters during inference
        output = network(data, data) 
        if isinstance(output, tuple):
            output = output[1]
            
        output = output.cpu().numpy()

        for (x, y, w, h), out in zip(indices, output):
            probs[x + w // 2, y + h // 2] += out

# Crop padded area
probs = probs[pad_size:image_w + pad_size, pad_size:image_h + pad_size, :]
pred_map = np.argmax(probs, axis=-1)

# Mask out unlabeled pixels (where ground truth is 0)
pred_map_masked = np.copy(pred_map)
pred_map_masked[gt_tar == 0] = 0

print('Inference complete.')


In [ ]:
# Calculate and plot Confusion Matrix
# We only evaluate pixels where gt_tar > 0
valid_mask = gt_tar > 0
y_true = gt_tar[valid_mask] - 1  # 0-indexed for confusion matrix
y_pred = pred_map[valid_mask]

# Get metrics dictionary from utility
results = metrics(y_pred, y_true, n_classes=n_classes)

print(f'Overall Accuracy: {results["Accuracy"]:.2f}%')

# Plot Confusion Matrix Heatmap
plt.figure(figsize=(10, 8))
cm = results['Confusion_matrix']
sns.heatmap(cm, annot=True, fmt='g', cmap='Blues', 
            xticklabels=labels, yticklabels=labels)
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()


In [ ]:
# Visualize Classification Maps
plt.figure(figsize=(16, 8))

# Define a discrete colormap for HSI
cmap = plt.get_cmap('jet', n_classes + 1)

plt.subplot(1, 2, 1)
plt.title('Predicted Classification Map')
plt.imshow(pred_map_masked, cmap=cmap, vmin=0, vmax=n_classes)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title('Ground Truth Map')
plt.imshow(gt_tar, cmap=cmap, vmin=0, vmax=n_classes)
plt.axis('off')

plt.tight_layout()
plt.show()
